# Advanced Kubeflow Pipelines: Monitoring and Debugging

### What you will learn
- How to implement pipeline monitoring and observability.
- Debugging failed pipeline runs using logs and metadata.
- Collecting custom metrics and configuring logging for pipeline components.
- Implementing error handling and retry logic in KFP pipelines.

### Why this matters in MLOps
In production, pipelines fail — data changes, resources become unavailable, code has bugs. Without proper monitoring and debugging tooling, diagnosing failures becomes a manual, time-consuming process. Observability is what separates a production ML system from a research prototype.

### You're done when...
- Pipeline components have structured logging configured.
- Custom metrics are collected and reported.
- Error handling and retry logic are implemented.

## The MLOps Perspective
Monitoring and debugging are the "eyes and ears" of your ML system. In traditional software, you monitor request latency and error rates. In ML pipelines, you must also monitor data drift, model performance degradation, and resource utilization patterns. Proactive monitoring — with structured logging, metrics collection, and automated alerting — enables teams to detect and resolve issues before they impact users.

## Setup and Imports

In [ ]:
# Install requirements
!pip install -r requirements.txt

In [ ]:
# Import required libraries
import kfp
from kfp import dsl
from kfp.dsl import component
import logging
import json
import time
from datetime import datetime
from typing import Optional

## Structured Logging in Pipeline Components
Python's logging module works inside KFP components. Structured logging (JSON format) is recommended for production as it integrates with log aggregation systems like Elasticsearch, Loki, or CloudWatch.

In [ ]:
# Configure structured logging for pipeline components
def setup_logger(name: str, level: int = logging.INFO) -> logging.Logger:
    """Configure a structured logger for KFP components."""
    logger = logging.getLogger(name)
    logger.setLevel(level)
    handler = logging.StreamHandler()
    handler.setFormatter(logging.Formatter(
        '{"timestamp": "%(asctime)s", "name": "%(name)s", '
        '"level": "%(levelname)s", "message": "%(message)s"}'
    ))
    if not logger.handlers:
        logger.addHandler(handler)
    return logger

In [ ]:
# Define components with structured logging
@component(
    base_image="python:3.11",
    packages_to_install=["numpy"]
)
def logged_data_ingestion(source: str) -> str:
    """Ingest data with structured logging."""
    logger = setup_logger("data_ingestion")
    logger.info(f"Starting data ingestion from {source}")

    import numpy as np
    try:
        data_shape = (1000, 20)
        logger.info(f"Data loaded successfully", extra={
            "shape": data_shape,
            "source": source
        })
        output_path = "/tmp/ingested_data.npy"
        np.random.randn(*data_shape).tofile(output_path)
        logger.info(f"Data saved to {output_path}")
        return output_path
    except Exception as e:
        logger.error(f"Data ingestion failed: {str(e)}")
        raise


@component(
    base_image="python:3.11",
    packages_to_install=["numpy"]
)
def logged_model_training(data_path: str, epochs: int) -> dict:
    """Train model with structured logging and metrics."""
    logger = setup_logger("model_training")
    logger.info(f"Starting training with {epochs} epochs")

    import numpy as np
    try:
        accuracy = float(np.random.uniform(0.80, 0.95))
        loss = float(np.random.uniform(0.05, 0.20))
        logger.info(f"Training completed", extra={
            "accuracy": accuracy,
            "loss": loss,
            "epochs": epochs
        })
        return {
            "accuracy": accuracy,
            "loss": loss,
            "epochs": epochs
        }
    except Exception as e:
        logger.error(f"Training failed: {str(e)}")
        raise

In [ ]:
# Pipeline with structured logging
@dsl.pipeline(
    name="monitored-training-pipeline",
    description="Pipeline with structured logging and observability"
)
def monitored_training_pipeline(
    data_source: str = "s3://data/raw",
    epochs: int = 10
):
    ingest = logged_data_ingestion(source=data_source)
    train = logged_model_training(
        data_path=ingest.output,
        epochs=epochs
    )

## Custom Metrics Collection
KFP allows logging custom metrics during pipeline execution. These metrics are automatically tracked and can be viewed in the KFP UI.

In [ ]:
# Define components that report custom metrics
from kfp.dsl import Metrics, Output

@component(
    base_image="python:3.11",
    packages_to_install=["numpy"]
)
def compute_drift_metrics(
    reference_path: str,
    current_path: str,
    metrics_output: Output[Metrics]
) -> dict:
    """Compute data drift metrics and log them."""
    import numpy as np
    import json

    reference = np.random.randn(1000)
    current = np.random.randn(1000) * 1.1  # Slightly shifted

    ps_value = float(np.random.uniform(0.01, 0.50))
    drift_score = float(np.random.uniform(0.0, 0.8))

    metrics_output.log_metric("psi", ps_value)
    metrics_output.log_metric("drift_score", drift_score)
    metrics_output.log_metric("feature_count", 20)
    metrics_output.log_metric("sample_count", len(reference))

    print(f"Drift metrics computed: PSI={ps_value:.4f}, Drift={drift_score:.4f}")
    return {"psi": ps_value, "drift_score": drift_score}


@component(
    base_image="python:3.11",
    packages_to_install=["numpy"]
)
def compute_model_metrics(
    predictions_path: str,
    ground_truth_path: str,
    metrics_output: Output[Metrics]
) -> dict:
    """Compute model performance metrics."""
    import numpy as np

    accuracy = float(np.random.uniform(0.75, 0.98))
    precision = float(np.random.uniform(0.70, 0.96))
    recall = float(np.random.uniform(0.70, 0.96))
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    metrics_output.log_metric("accuracy", accuracy)
    metrics_output.log_metric("precision", precision)
    metrics_output.log_metric("recall", recall)
    metrics_output.log_metric("f1_score", f1)

    print(f"Model metrics: accuracy={accuracy:.4f}, f1={f1:.4f}")
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    }


@dsl.pipeline(
    name="metrics-collection-pipeline",
    description="Pipeline that collects and reports custom metrics"
)
def metrics_collection_pipeline(
    reference_data: str = "data/reference/",
    current_data: str = "data/current/",
    predictions: str = "data/predictions/",
    ground_truth: str = "data/ground_truth/"
):
    drift = compute_drift_metrics(
        reference_path=reference_data,
        current_path=current_data
    )
    model_perf = compute_model_metrics(
        predictions_path=predictions,
        ground_truth_path=ground_truth
    )

## Error Handling and Retry Logic
Production pipelines must handle failures gracefully. KFP provides mechanisms for retrying failed components and implementing custom error handling.

In [ ]:
# Define components with error handling and retry logic
@component(
    base_image="python:3.11",
    packages_to_install=["requests"]
)
def unreliable_api_call(endpoint: str, max_retries: int = 3) -> dict:
    """Simulate an API call with retry logic."""
    import random
    import time

    for attempt in range(max_retries):
        try:
            success = random.random() > 0.4  # 60% success rate
            if success:
                result = {
                    "status": "success",
                    "endpoint": endpoint,
                    "attempt": attempt + 1,
                    "data": {"value": random.randint(1, 100)}
                }
                print(f"API call succeeded on attempt {attempt + 1}: {result}")
                return result
            else:
                raise ConnectionError(f"Simulated failure on attempt {attempt + 1}")
        except ConnectionError as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt  # Exponential backoff
                print(f"Retrying in {wait_time}s...")
                time.sleep(wait_time)
            else:
                print(f"All {max_retries} attempts failed")
                return {"status": "failed", "endpoint": endpoint, "error": str(e)}


@component(
    base_image="python:3.11"
)
def fallback_processing(failed_endpoint: str) -> str:
    """Fallback processing when primary source fails."""
    fallback_path = f"/data/fallback/{datetime.now().strftime('%Y%m%d')}"
    print(f"Using fallback data path: {fallback_path}")
    return fallback_path

In [ ]:
# Pipeline with error handling and retry using KFP's built-in retry
@dsl.pipeline(
    name="resilient-pipeline",
    description="Pipeline with retry logic and error handling"
)
def resilient_pipeline(endpoint: str = "https://api.example.com/data"):
    api_task = unreliable_api_call(endpoint=endpoint, max_retries=3)
    api_task.set_caching_options(enable_caching=False)
    api_task.set_retry(num_retries=2)

    with dsl.Condition(api_task.output["status"] == "failed", name="fallback-path"):
        fallback = fallback_processing(failed_endpoint=endpoint)
        print(f"Fallback triggered: {fallback.output}")

## Monitoring Dashboard Configuration
KFP pipeline metrics can be exported to monitoring systems like Prometheus and visualized in dashboards.

In [ ]:
# Define a monitoring configuration component
@component(
    base_image="python:3.11",
    packages_to_install=["json"]
)
def generate_monitoring_config(
    pipeline_name: str,
    metrics_list: list,
    alert_thresholds: dict
) -> str:
    """Generate a monitoring dashboard configuration."""
    import json

    config = {
        "pipeline": pipeline_name,
        "metrics": metrics_list,
        "alerts": [
            {
                "metric": metric,
                "threshold": threshold,
                "action": "notify",
                "channel": "slack"
            }
            for metric, threshold in alert_thresholds.items()
        ],
        "dashboard": {
            "title": f"{pipeline_name} - Monitoring Dashboard",
            "refresh_interval": "5m"
        }
    }
    config_path = "/tmp/monitoring_config.json"
    with open(config_path, "w") as f:
        json.dump(config, f, indent=2)
    print(f"Monitoring configuration generated: {json.dumps(config, indent=2)}")
    return config_path


@dsl.pipeline(
    name="monitoring-dashboard-pipeline",
    description="Pipeline that generates monitoring dashboard configuration"
)
def monitoring_dashboard_pipeline():
    config = generate_monitoring_config(
        pipeline_name="production-retraining",
        metrics_list=["accuracy", "f1_score", "drift_score"],
        alert_thresholds={
            "accuracy": 0.80,
            "f1_score": 0.75,
            "drift_score": 0.3
        }
    )

## Debugging Failed Pipeline Runs
When pipelines fail, KFP provides several debugging tools: pod logs, artifact inspection, and run metadata.

In [ ]:
# Demonstrate debugging with KFP client
def debug_pipeline_run(kfp_client, run_id: str) -> dict:
    """Debug a pipeline run by inspecting logs and metadata."""
    try:
        run_detail = kfp_client.get_run(run_id=run_id)
        status = run_detail.state
        print(f"Run {run_id} status: {status}")

        if status == "FAILED":
            error_info = run_detail.error
            print(f"Error details: {error_info}")

        return {
            "run_id": run_id,
            "status": status,
            "error": getattr(run_detail, 'error', None)
        }
    except Exception as e:
        print(f"Could not fetch run details: {e}")
        return {"run_id": run_id, "error": str(e)}


@component(
    base_image="python:3.11",
    packages_to_install=["kfp"]
)
def inspect_run_artifacts(run_id: str) -> dict:
    """Inspect artifacts produced by a pipeline run."""
    import kfp
    client = kfp.Client()
    try:
        run = client.get_run(run_id=run_id)
        artifacts = {
            "run_id": run_id,
            "status": run.state,
            "pipeline_name": run.pipeline_spec.pipeline_name if hasattr(run, 'pipeline_spec') else "unknown"
        }
        print(f"Run artifacts: {artifacts}")
        return artifacts
    except Exception as e:
        print(f"Error inspecting artifacts: {e}")
        return {"run_id": run_id, "error": str(e)}

## Fill-in-the-Blank Exercises

In [ ]:
# Exercise 1: Complete the retry configuration for a pipeline component
@dsl.pipeline(name="exercise-retry")
def exercise_retry_pipeline():
    component = unreliable_api_call(endpoint="https://api.example.com")
    component.___(num_retries=___)  # Set 3 retries
# Hint: Use the method set_retry with appropriate retry count

In [ ]:
# Exercise 2: Complete the metrics logging
from kfp.dsl import Metrics, Output

@component(base_image="python:3.11")
def exercise_metrics_collection(metrics_output: Output[Metrics]) -> None:
    """Log training metrics to KFP."""
    metrics_output.log_metric("accuracy", 0.93)
    metrics_output.log_metric("___", ___)  # Add an f1_score of 0.89
    metrics_output.log_metric("training_duration_seconds", 120.5)
    print("Metrics logged successfully")
# Hint: log_metric takes a metric name string and a float value

## Summary
In this notebook, you learned:
- Configuring structured logging for pipeline observability
- Collecting and reporting custom metrics using KFP's Metrics output
- Implementing retry logic and error handling for resilient pipelines
- Generating monitoring dashboard configurations
- Debugging failed pipeline runs through the KFP API
- These practices ensure production pipelines are observable, debuggable, and resilient